<a href="https://colab.research.google.com/github/peteraggi/ColabFavourites/blob/main/CoffeeCare_Dataset_Builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CoffeeCare Dataset Builder
Builds the CoffeeCare coffee leaf disease dataset from five public datasets plus an `unknown` class.

**All images are kept**, including the augmented (rotated/flipped) copies the dataset authors added.
Copies of the same leaf are only grouped, so they land in the same split (train, val or test).

| Dataset | Country | Species | Role |
|---|---|---|---|
| Soroti University (Chelangat et al., 2025) | Uganda | not stated | test_uganda (never trained on) |
| JMuBEN / JMuBEN2 (Jepkoech et al., 2021) | Kenya | Arabica | train_pool |
| BRACOL (Krohling et al., 2019) | Brazil | Arabica | train_pool |
| RoCoLe (Parraga-Alava et al., 2019) | Ecuador | Robusta | train_pool |
| beans, cassava (Makerere AI Lab), PlantVillage | Uganda / various | not coffee | train_pool, label `unknown` |

In [1]:
# Let Put all imports in one place
import os
import shutil
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

print('numpy', np.__version__)
print('pandas', pd.__version__)

numpy 2.4.6
pandas 2.2.3


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Permanent storage on Google Drive
BASE_DIR = '/content/drive/MyDrive/ColabFavourites/CoffeeCare'
RAW_DIR = os.path.join(BASE_DIR, 'raw')
DATASET_DIR = os.path.join(BASE_DIR, 'dataset')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
METADATA_CSV = os.path.join(BASE_DIR, 'metadata.csv')

# Fast temporary storage on Colab (wiped when the session ends)
EXTRACTED_DIR = '/content/extracted'

for folder in [RAW_DIR, DATASET_DIR, MODELS_DIR, EXTRACTED_DIR]:
    os.makedirs(folder, exist_ok=True)
    print(f"Ready: {folder}")

print()
print('Already on Drive:')
for name in sorted(os.listdir(BASE_DIR)):
    path = os.path.join(BASE_DIR, name)
    if os.path.isfile(path):
        size_mb = round(os.path.getsize(path) / 1024 / 1024, 1)
        print(f'   {name}  ({size_mb} MB)')

print()
print('Downloaded zips in raw/:')
for name in sorted(os.listdir(RAW_DIR)):
    size_mb = round(os.path.getsize(os.path.join(RAW_DIR, name)) / 1024 / 1024, 1)
    print(f'   {name}  ({size_mb} MB)')

Ready: /content/drive/MyDrive/ColabFavourites/CoffeeCare/raw
Ready: /content/drive/MyDrive/ColabFavourites/CoffeeCare/dataset
Ready: /content/drive/MyDrive/ColabFavourites/CoffeeCare/models
Ready: /content/extracted

Already on Drive:
   broken_images.csv  (0.0 MB)
   broken_images_kenya.csv  (0.0 MB)
   image_hashes.csv  (11.8 MB)
   label_conflicts.csv  (0.2 MB)
   metadata.csv  (15.0 MB)
   splits_v1.csv  (0.6 MB)

Downloaded zips in raw/:
   bracol.zip  (156.9 MB)
   jmuben.zip  (234.4 MB)
   jmuben2.zip  (453.4 MB)
   rocole.zip  (2141.6 MB)
   soroti_uganda.zip  (24.8 MB)


In [4]:
# The 7 CoffeeCare classes (same keys as diseases.json in the backend, alphabetical)
COFFEECARE_CLASSES = [
    'brown_eye_spot',
    'coffee_leaf_rust',
    'healthy',
    'leaf_miner',
    'phoma_leaf_blight',
    'red_spider_mite',
    'unknown'
]

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png')
SEED = 42

# Dataset build settings
KEEP_COPIES_TOGETHER = True   # copies of one leaf stay in the same split (no image is removed)
MAX_TRAIN_PER_CLASS = None    # None = keep every training image; a number (e.g. 3000) caps big classes
DATASET_VERSION = 'v1'

print('Classes:', len(COFFEECARE_CLASSES))

Classes: 7


In [5]:
def download_dataset(name, url=None):
    """Use the zip already on Drive, or download it if it is missing"""
    zip_path = os.path.join(RAW_DIR, f'{name}.zip')

    if os.path.exists(zip_path) and zipfile.is_zipfile(zip_path):
        print('Already downloaded:', zip_path)
    else:
        if url is None:
            raise ValueError(f'{name}: not on Drive yet. Give its download link.')
        !wget -q --show-progress -O "{zip_path}" "{url}"

    if not zipfile.is_zipfile(zip_path):
        os.remove(zip_path)
        raise ValueError(f'{name}: download is not a valid zip. Copy the link again.')

    size_mb = round(os.path.getsize(zip_path) / 1024 / 1024, 1)
    print(f'{name}: {size_mb} MB')
    return zip_path


def extract_dataset(name, zip_path):
    """Unzip to fast local disk, including any zip files inside the zip"""
    target = os.path.join(EXTRACTED_DIR, name)

    if os.path.exists(target):
        print('Already extracted:', target)
        return target

    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(target)

        # Some datasets contain more zip files inside: unzip those too
        found_inner = True
        while found_inner:
            found_inner = False
            for current, folders, files in os.walk(target):
                for filename in files:
                    if filename.lower().endswith('.zip'):
                        inner_zip = os.path.join(current, filename)
                        inner_target = os.path.join(current, filename[:-4])
                        with zipfile.ZipFile(inner_zip, 'r') as z:
                            z.extractall(inner_target)
                        os.remove(inner_zip)
                        found_inner = True
                        print('   unzipped inner file:', filename)

    except zipfile.BadZipFile as e:
        shutil.rmtree(target, ignore_errors=True)
        raise ValueError(f'{name}: a zip could not be opened ({e}). Partial folder removed.')

    print('Extracted to', target)
    return target

print('Functions ready')

Functions ready


In [6]:
def show_tree(root, max_depth=4):
    """Print the folder structure with the number of images in each folder"""
    for current, folders, files in os.walk(root):
        depth = current[len(root):].count(os.sep)
        if depth > max_depth:
            continue

        images = [f for f in files if f.lower().endswith(IMAGE_EXTENSIONS)]
        others = [f for f in files if not f.lower().endswith(IMAGE_EXTENSIONS)]

        indent = '    ' * depth
        print(f"{indent}{os.path.basename(current)}/  -> {len(images)} images, {len(others)} other files")

        if others and len(others) <= 5:
            print(f"{indent}    other files: {others}")


def find_unmapped(root, class_map):
    """List folders that contain images but are not in the class map"""
    unmapped = []
    for current, folders, files in os.walk(root):
        images = [f for f in files if f.lower().endswith(IMAGE_EXTENSIONS)]
        folder_name = os.path.basename(current).strip().lower()
        if images and folder_name not in class_map:
            unmapped.append((current, len(images)))
    return unmapped

print('Inspection functions ready')

Inspection functions ready


In [7]:
def collect_images(source, root, class_map, species, role):
    """Label images by their folder name and return one row per image"""
    rows = []
    for current, folders, files in os.walk(root):
        folder_name = os.path.basename(current).strip().lower()
        if folder_name not in class_map:
            continue

        label = class_map[folder_name]

        for filename in sorted(files):
            if not filename.lower().endswith(IMAGE_EXTENSIONS):
                continue

            full_path = os.path.join(current, filename)
            rows.append({
                'relative_path': os.path.relpath(full_path, EXTRACTED_DIR),
                'source': source,
                'original_class': os.path.basename(current),
                'label': label,
                'species': species,
                'role': role
            })
    return rows


def collect_from_table(source, root, table, file_column, label_column, class_map, species, role):
    """Label images using a spreadsheet (file name + label columns) and return one row per image"""

    # 1. Find every image under root, by file name
    image_paths = {}
    for current, folders, files in os.walk(root):
        for filename in files:
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                image_paths[filename] = os.path.join(current, filename)

    # 2. Match each spreadsheet row to its image
    rows = []
    missing_files = []
    unmapped_labels = set()

    for _, record in table.iterrows():
        filename = str(record[file_column]).strip()
        original_label = str(record[label_column]).strip()

        if filename not in image_paths:
            missing_files.append(filename)
            continue

        if original_label not in class_map:
            unmapped_labels.add(original_label)
            continue

        rows.append({
            'relative_path': os.path.relpath(image_paths[filename], EXTRACTED_DIR),
            'source': source,
            'original_class': original_label,
            'label': class_map[original_label],
            'species': species,
            'role': role
        })

    print(f'{source}: {len(rows)} images matched')
    print('   rows with no image file:', len(missing_files), missing_files[:5])
    print('   labels not in class map:', unmapped_labels)
    return rows

print('Labelling functions ready')

Labelling functions ready


In [8]:
def check_images(df):
    """Count image sizes and find images that cannot be opened"""
    sizes = {}
    broken = []

    for relative_path in tqdm(df['relative_path'], desc='Checking images'):
        try:
            with Image.open(os.path.join(EXTRACTED_DIR, relative_path)) as img:
                sizes[img.size] = sizes.get(img.size, 0) + 1
        except Exception:
            broken.append(relative_path)

    print('Image sizes (width, height) -> count:', sizes)
    print('Broken images:', len(broken))
    return broken


def clean_rows(rows, name):
    """Turn rows into a table, remove images that cannot be opened, and save the list of removed files"""
    table = pd.DataFrame(rows)
    broken = check_images(table)

    table[table['relative_path'].isin(broken)].to_csv(
        os.path.join(BASE_DIR, f'broken_images_{name}.csv'), index=False)

    table = table[~table['relative_path'].isin(broken)].reset_index(drop=True)
    print(f'{name}: {len(table)} usable images')
    print(table['label'].value_counts())
    return table


def show_samples(df, per_class=4):
    """Show a few random images from each class"""
    labels = sorted(df['label'].unique())
    fig, axes = plt.subplots(len(labels), per_class,
                             figsize=(3 * per_class, 3 * len(labels)),
                             squeeze=False)

    for i, label in enumerate(labels):
        class_df = df[df['label'] == label]
        sample = class_df.sample(min(per_class, len(class_df)), random_state=SEED)

        for j, relative_path in enumerate(sample['relative_path']):
            img = Image.open(os.path.join(EXTRACTED_DIR, relative_path))
            axes[i, j].imshow(img)
            axes[i, j].set_title(label, fontsize=9)

        for j in range(per_class):
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

print('Cleaning and viewing functions ready')

Cleaning and viewing functions ready


## Soroti University (Uganda): test set only
Mendeley: https://data.mendeley.com/datasets/k36wnd6knb/1
The published archive contains 102 zero-byte (empty) files, which are removed.
Role = test_uganda: never used for training.